# Steering Manifold Repair — denoiser + DPAR experiment suite

This notebook assumes the **sentiment additive baseline already passed**. It does not change the frozen baseline prompts, seeds, judge, vector, or intervention layer.

Hypotheses tested:

1. **H1 — Gaussian denoiser:** the assignment-proposed activation denoiser improves the concept/fluency frontier.
2. **H2 — DPAR:** vanilla denoising partly cancels the intended steering direction; removing the correction component parallel to the steering vector preserves effective $\alpha$ and can improve the frontier.
3. **H3 — Structured corruption:** training on a mixture of Gaussian noise and natural activation-difference directions gives a better steering repair prior than isotropic Gaussian noise alone.
4. **Ablations:** norm-preserving repair and $\lambda=0.5$ partial direction preservation.

The validation sentiment direction is **never used to train either denoiser**.

In [ ]:
# Clone once in a fresh Colab. If already cloned, this cell pulls the latest main.
import os, subprocess, pathlib
repo = pathlib.Path('/content/steering-manifold-repair')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/Nek1tt/steering-manifold-repair.git', str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
os.chdir(repo)
print('cwd:', os.getcwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.'], check=True)
subprocess.run([sys.executable,'-m','pytest','-q'], check=True)

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Switch Colab runtime to GPU before training.'


## 0. Safety check: frozen successful baseline

The repair suite expects `results/sentiment_direction.pt`, produced by the already-validated sentiment baseline. If this is a fresh runtime, rerun the fast direction validation first; this is **not** a new baseline search, it reconstructs the same frozen direction from committed matched examples and held-out calibration prompts.

In [ ]:
from pathlib import Path
import subprocess, sys
if not Path('results/sentiment_direction.pt').exists():
    subprocess.run([sys.executable,'scripts/validate_sentiment_baseline.py','--config','configs/baseline_sentiment_gpt2.yaml'], check=True)
else:
    print('Using existing frozen direction:', Path('results/sentiment_direction.pt'))

## 1. Cache generic natural activations

Uses WikiText-2 only as **generic LM text**. We cache up to 80k natural `blocks.6.hook_resid_post` activation vectors. Sentiment evaluation prompts and the steering vector are not part of this data.

In [ ]:
subprocess.run([sys.executable,'scripts/cache_activations.py','--config','configs/repair_suite_gpt2.yaml'], check=True)

## 2. Train B2 — Gaussian denoiser

Corruption magnitude is sampled log-uniformly and normalized by $\|h\|$ so one network covers weak through severe interventions. The MLP is residual and starts as identity.

In [ ]:
subprocess.run([sys.executable,'scripts/train_denoiser.py','--config','configs/repair_suite_gpt2.yaml','--kind','gaussian'], check=True)

## 3. Train M1 — mixed structured denoiser

Half of non-identity corruptions use a normalized random natural-activation difference direction $h_j-h_k$; the rest use Gaussian noise. This tests whether anisotropic structured perturbations transfer better to an unseen steering direction.

In [ ]:
subprocess.run([sys.executable,'scripts/train_denoiser.py','--config','configs/repair_suite_gpt2.yaml','--kind','mixed'], check=True)

## 4. Evaluate all repair hypotheses on the frozen baseline

Methods:

- `additive` — $h+\alpha v$
- `norm_preserving` — cheap non-learned control
- `gaussian` — assignment-proposed denoiser
- `gaussian_lambda05` — partial direction preservation
- `gaussian_dpar` — Direction-Preserving Activation Repair
- `mixed` — structured-corruption denoiser
- `mixed_dpar` — full proposed method

Main grid: $\alpha\in\{0,0.5,0.75,1,1.5,2,3,4\}$.

In [ ]:
subprocess.run([sys.executable,'scripts/eval_repairs.py','--config','configs/repair_suite_gpt2.yaml'], check=True)

## 5. Pareto + mechanistic diagnostics

In addition to the Pareto plot, the code records:

- requested vs effective $\alpha$ after repair;
- cosine between repair correction and steering vector;
- fraction of correction norm parallel to the steering vector;
- correction norm relative to the steering perturbation.

This directly checks whether a vanilla denoiser improves fluency by cancelling steering.

In [ ]:
subprocess.run([sys.executable,'scripts/plot_repairs.py','--config','configs/repair_suite_gpt2.yaml'], check=True)

In [ ]:
from IPython.display import display, Image, Markdown
from pathlib import Path
for name in ['repair_pareto.png','effective_alpha.png','correction_geometry.png']:
    p = Path('results/repair_suite')/name
    if p.exists():
        display(Image(filename=str(p)))
report = Path('results/repair_suite/hypothesis_report.md')
if report.exists():
    display(Markdown(report.read_text()))

In [ ]:
import pandas as pd
agg = pd.read_csv('results/repair_suite/repair_aggregate.csv')
frontier = pd.read_csv('results/repair_suite/frontier_summary.csv')
display(frontier)
cols = ['method','strength','fluency_score','concept_score','effective_alpha','alpha_preservation_error','correction_cos_v','correction_parallel_fraction']
display(agg[cols].sort_values(['method','strength']))

## Interpretation checklist

A useful result does **not** require every hypothesis to win.

- If Gaussian repair improves fluency but lowers effective $\alpha$, that is evidence for the cancellation failure mode.
- If DPAR keeps `effective_alpha ≈ requested alpha` while recovering some fluency, H2 is supported.
- If `mixed_dpar` beats `gaussian_dpar` at matched concept thresholds, H3 is supported.
- A negative H3 is still informative: the simplest geometric projection may matter more than the corruption prior.

Do not retune the frozen baseline after seeing repair results.